# Random forest // Ensemble

### heart deseases dataset

In [8]:
import pandas as pd
import numpy as np

URL = "https://gist.githubusercontent.com/notreallyme2/8d20f6b5bcc3606e0541cbdf2ee0a7a6/raw/heart.csv" # heart deseases dataset

df = pd.read_csv(URL)

print(df.head())
print(df.shape)
print(df.describe())
print(df.info())

# print(df.isnull().sum()) # 0 missing values
# print(df.duplicated().sum()) # 1 duplicate
# df = df.drop_duplicates()

   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   3       145   233    1        0      150      0      2.3      0   
1   37    1   2       130   250    0        1      187      0      3.5      0   
2   41    0   1       130   204    0        0      172      0      1.4      2   
3   56    1   1       120   236    0        1      178      0      0.8      2   
4   57    0   0       120   354    0        1      163      1      0.6      2   

   ca  thal  target  
0   0     1       1  
1   0     2       1  
2   0     2       1  
3   0     2       1  
4   0     2       1  
(303, 14)
              age         sex          cp    trestbps        chol         fbs  \
count  303.000000  303.000000  303.000000  303.000000  303.000000  303.000000   
mean    54.366337    0.683168    0.966997  131.623762  246.264026    0.148515   
std      9.082101    0.466011    1.032052   17.538143   51.830751    0.356198   
min     29.000000    0.000000    0.000000   94.

### baseline model

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X = df.drop(columns="target")
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

print(rf.score(X_train, y_train)) # 1.0
print(rf.score(X_test, y_test)) # 0.819672131147541
# this baseline model

1.0
0.819672131147541


### prepruned model

In [17]:
from sklearn.model_selection import GridSearchCV

X = df.drop(columns="target")
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
rf = RandomForestClassifier(max_depth=2, random_state=42, n_estimators=100)
rf.fit(X_train, y_train)

print(rf.score(X_train, y_train))
print(rf.score(X_test, y_test))
# 0.8589211618257261
# 0.7704918032786885
# became better but still not the best



# grid search to find the best combination of prepruned parameters
grid = {
    'max_depth': [2, 3, 4, 5, 6, 7],
    'n_estimators': [3, 5,10, 50, 100],
    'max_features': [0.5, 'sqrt', None],
}

gs = GridSearchCV(
    RandomForestClassifier(),
    param_grid=grid, cv=5, n_jobs=-1
)
gs.fit(X_train, y_train)

print("Best Parameters:", gs.best_params_)
print("Best Cross-Validation Score during search:", gs.best_score_)
# {'max_depth': 3, 'max_features': 'sqrt', 'n_estimators': 50}
# 0.8592687074829932

model = RandomForestClassifier(max_depth=3, random_state=42, n_estimators=50, n_jobs=-1, max_features='sqrt')
model.fit(X_train, y_train)
print(model.score(X_train, y_train)) # 0.8921161825726142
print(model.score(X_test, y_test)) # 0.8360655737704918

0.8589211618257261
0.7704918032786885
Best Parameters: {'max_depth': 3, 'max_features': 0.5, 'n_estimators': 50}
Best Cross-Validation Score during search: 0.8467687074829933
0.8921161825726142
0.8360655737704918


### feature importance

In [14]:
# need to take only influential features

imp = pd.Series(gs.best_estimator_.feature_importances_, index=X.columns).sort_values(ascending=False)
print(imp)

# cp          0.255897
# thal        0.140846
# oldpeak     0.139689
# thalach     0.120926
# best 4 parameters

cp          0.255897
thal        0.140846
oldpeak     0.139689
thalach     0.120926
exang       0.082054
ca          0.079744
slope       0.041794
age         0.040765
chol        0.039653
trestbps    0.030202
sex         0.022070
restecg     0.006076
fbs         0.000284
dtype: float64


### model with influential features

In [16]:
X = df[['cp', 'thal', 'oldpeak', 'thalach']]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# grid = {
#     'max_depth': [2, 3, 4, 5, 6, 7],
#     'n_estimators': [3, 5,10, 50, 100],
#     'max_features': [0.5, 'sqrt', None],
# }
#
# gs = GridSearchCV(
#     RandomForestClassifier(),
#     param_grid=grid, cv=5, n_jobs=-1
# )
# gs.fit(X_train, y_train)
# print("Best Parameters:", gs.best_params_)
# print("Best Cross-Validation Score during search:", gs.best_score_)
#
# {'max_depth': 2, 'max_features': 'sqrt', 'n_estimators': 100}
# model prepruning uchun best parameters topildi

model = RandomForestClassifier(max_depth=2, random_state=42, n_estimators=100, n_jobs=-1, max_features='sqrt')
model.fit(X_train, y_train)
print(model.score(X_train, y_train)) # 0.8381742738589212
print(model.score(X_test, y_test)) # 0.7868852459016393

Best Parameters: {'max_depth': 2, 'max_features': 'sqrt', 'n_estimators': 100}
Best Cross-Validation Score during search: 0.8383503401360544
0.8381742738589212
0.7868852459016393


### conclusion

In [19]:
# All features: 0.8360655737704918
# Selected features: 0.7868852459016393

# Conclusion
#
# Using only 4 selected features reduced test accuracy from 83.6% to 78.7%. Although using fewer features can reduce memory usage and training time, in this case it also reduced the model's performance. So, using fewer features is not always better, and we need to find a balance between model complexity and accuracy.